# Module 3c: Ray Train - Distributed Training

**DSC 232R - Big Data Analysis Using Spark**

This notebook covers Ray Train:
1. XGBoost distributed training
2. Scaling configuration
3. Checkpointing and results
4. Weather data prediction example

**Prerequisites**: `pip install "ray[train]" xgboost`

## Key Takeaways

- **Ray Train** provides distributed training for XGBoost, PyTorch, TensorFlow
- **ScalingConfig** controls workers and resources
- **Trainers** handle data distribution and result collection
- Direct integration with Ray Data for end-to-end pipelines

In [ ]:
import ray
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Initialize Ray
if ray.is_initialized():
    ray.shutdown()

ray.init(num_cpus=4, logging_level="WARNING")
print(f"Ray version: {ray.__version__}")

---

## 1. XGBoost with Ray Train

### Connecting to DSC 232R

You've used XGBoost in Class13-15. Ray Train lets you scale XGBoost training across multiple workers.

In [ ]:
# Generate sample regression data
X, y = make_regression(n_samples=10000, n_features=20, noise=0.1, random_state=42)

# Create pandas DataFrame
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

print(f"Dataset shape: {df.shape}")
print(f"Features: {feature_names[:5]}...")

In [ ]:
# Convert to Ray Dataset
dataset = ray.data.from_pandas(df)

# Split into train/test
train_dataset, valid_dataset = dataset.train_test_split(test_size=0.2)

print(f"Train dataset: {train_dataset.count()} rows")
print(f"Valid dataset: {valid_dataset.count()} rows")

### Basic XGBoostTrainer

In [ ]:
from ray.train.xgboost import XGBoostTrainer
from ray.train import ScalingConfig

# Define trainer
trainer = XGBoostTrainer(
    label_column="target",
    num_boost_round=50,
    params={
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 6,
        "eta": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
    datasets={"train": train_dataset, "valid": valid_dataset},
    scaling_config=ScalingConfig(
        num_workers=2,  # Distributed across 2 workers
        use_gpu=False,
    ),
)

print("Trainer configured.")
print(f"  Scaling: {trainer.scaling_config}")

In [ ]:
# Train the model
result = trainer.fit()

print("\nTraining Results:")
print(f"  Final RMSE (valid): {result.metrics.get('valid-rmse', 'N/A')}")
print(f"  Checkpoint: {result.checkpoint}")

### Understanding ScalingConfig

In [ ]:
# Different scaling configurations

scaling_configs = {
    "single_worker": ScalingConfig(
        num_workers=1,
        use_gpu=False,
    ),
    "multi_worker_cpu": ScalingConfig(
        num_workers=4,
        use_gpu=False,
        resources_per_worker={"CPU": 2},
    ),
    "multi_worker_gpu": ScalingConfig(
        num_workers=4,
        use_gpu=True,
        resources_per_worker={"GPU": 1},
    ),
}

print("Scaling Configuration Options:")
print("=" * 60)
for name, config in scaling_configs.items():
    print(f"\n{name}:")
    print(f"  Workers: {config.num_workers}")
    print(f"  Use GPU: {config.use_gpu}")
    print(f"  Resources: {config.resources_per_worker}")

---

## 2. Using the Trained Model

### Loading from Checkpoint

In [ ]:
import xgboost as xgb

# Load model from checkpoint
checkpoint = result.checkpoint

# Get the booster
with checkpoint.as_directory() as checkpoint_dir:
    import os
    model_path = os.path.join(checkpoint_dir, "model.ubj")
    if os.path.exists(model_path):
        model = xgb.Booster()
        model.load_model(model_path)
        print("Model loaded from checkpoint.")
    else:
        print(f"Available files: {os.listdir(checkpoint_dir)}")

### Making Predictions

In [ ]:
# Batch prediction using Ray Data
from ray.train.xgboost import XGBoostPredictor
from ray.train.batch_predictor import BatchPredictor

# Create batch predictor
batch_predictor = BatchPredictor.from_checkpoint(
    result.checkpoint,
    XGBoostPredictor
)

# Prepare test data (remove target column)
test_features = valid_dataset.drop_columns(["target"])

# Make predictions
predictions = batch_predictor.predict(test_features)

print("Sample predictions:")
predictions.take(5)

---

## 3. Weather Data Prediction Example

Let's build a complete ML pipeline using weather data:

In [ ]:
# Generate synthetic weather data
np.random.seed(42)
n_samples = 20000

# Features
temperature = np.random.uniform(-10, 40, n_samples)  # Celsius
humidity = np.random.uniform(20, 100, n_samples)  # Percent
pressure = np.random.uniform(980, 1040, n_samples)  # hPa
wind_speed = np.random.uniform(0, 30, n_samples)  # m/s
cloud_cover = np.random.uniform(0, 100, n_samples)  # Percent
month = np.random.randint(1, 13, n_samples)
hour = np.random.randint(0, 24, n_samples)

# Target: precipitation (mm) - synthetic relationship
precipitation = (
    0.5 * (humidity / 100) * (cloud_cover / 100) * 10 +
    0.3 * np.maximum(0, (humidity - 60) / 40) * 5 +
    np.random.exponential(0.5, n_samples)
)
precipitation = np.clip(precipitation, 0, 50)

weather_df = pd.DataFrame({
    "temperature": temperature,
    "humidity": humidity,
    "pressure": pressure,
    "wind_speed": wind_speed,
    "cloud_cover": cloud_cover,
    "month": month,
    "hour": hour,
    "precipitation": precipitation,
})

print(f"Weather dataset shape: {weather_df.shape}")
print(f"\nSample:")
weather_df.head()

In [ ]:
# Create Ray Dataset and preprocess
ds_weather = ray.data.from_pandas(weather_df)

# Feature engineering
def add_features(batch: pd.DataFrame) -> pd.DataFrame:
    """Add derived features."""
    result = batch.copy()
    
    # Dew point approximation
    result["dew_point"] = result["temperature"] - ((100 - result["humidity"]) / 5)
    
    # Is daytime (6am - 6pm)
    result["is_daytime"] = ((result["hour"] >= 6) & (result["hour"] < 18)).astype(int)
    
    # Season encoding (simple)
    result["is_winter"] = result["month"].isin([12, 1, 2]).astype(int)
    result["is_summer"] = result["month"].isin([6, 7, 8]).astype(int)
    
    return result

ds_processed = ds_weather.map_batches(add_features, batch_format="pandas")

print("Processed dataset columns:")
print(ds_processed.schema())

In [ ]:
# Split data
train_ds, valid_ds = ds_processed.train_test_split(test_size=0.2)

print(f"Training samples: {train_ds.count()}")
print(f"Validation samples: {valid_ds.count()}")

In [ ]:
# Train XGBoost model for precipitation prediction
weather_trainer = XGBoostTrainer(
    label_column="precipitation",
    num_boost_round=100,
    params={
        "objective": "reg:squarederror",
        "eval_metric": ["rmse", "mae"],
        "max_depth": 8,
        "eta": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 3,
    },
    datasets={"train": train_ds, "valid": valid_ds},
    scaling_config=ScalingConfig(
        num_workers=2,
        use_gpu=False,
    ),
)

# Train
weather_result = weather_trainer.fit()

print("\nWeather Model Results:")
print(f"  Final RMSE: {weather_result.metrics.get('valid-rmse', 'N/A'):.4f}")
print(f"  Final MAE: {weather_result.metrics.get('valid-mae', 'N/A'):.4f}")

---

## 4. Classification Example

In [ ]:
# Generate classification data
X_cls, y_cls = make_classification(
    n_samples=5000,
    n_features=15,
    n_informative=10,
    n_classes=3,
    random_state=42
)

# Create DataFrame
cls_df = pd.DataFrame(X_cls, columns=[f"f{i}" for i in range(X_cls.shape[1])])
cls_df["label"] = y_cls

# Convert to Ray Dataset
cls_dataset = ray.data.from_pandas(cls_df)
cls_train, cls_valid = cls_dataset.train_test_split(test_size=0.2)

print(f"Classification dataset: {cls_df.shape}")
print(f"Classes: {np.unique(y_cls)}")

In [ ]:
# Train classifier
cls_trainer = XGBoostTrainer(
    label_column="label",
    num_boost_round=50,
    params={
        "objective": "multi:softmax",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "max_depth": 6,
        "eta": 0.1,
    },
    datasets={"train": cls_train, "valid": cls_valid},
    scaling_config=ScalingConfig(num_workers=2),
)

cls_result = cls_trainer.fit()

print("\nClassification Results:")
print(f"  Final mlogloss: {cls_result.metrics.get('valid-mlogloss', 'N/A'):.4f}")

---

## 5. Exercise: Complete ML Pipeline

Build a complete pipeline that:
1. Loads data
2. Engineers features
3. Trains a model
4. Evaluates results

In [ ]:
# Exercise: Build a pipeline to predict wind speed from other weather features

def build_wind_prediction_pipeline():
    """
    Build a pipeline to predict wind_speed from:
    - temperature
    - humidity  
    - pressure
    - cloud_cover
    - month
    - hour
    
    Steps:
    1. Create Ray Dataset from weather_df (drop precipitation and wind_speed as features)
    2. Add feature engineering (pressure gradient proxy, etc.)
    3. Split into train/valid
    4. Train XGBoostTrainer with label_column="wind_speed"
    5. Return results
    """
    # Your code here
    pass

# result = build_wind_prediction_pipeline()
# print(f"Wind prediction RMSE: {result.metrics.get('valid-rmse', 'N/A')}")

In [ ]:
# Solution

def build_wind_prediction_pipeline_solution():
    # Prepare data (use wind_speed as target, drop precipitation)
    wind_df = weather_df.drop(columns=["precipitation"])
    
    # Create Ray Dataset
    ds = ray.data.from_pandas(wind_df)
    
    # Feature engineering
    def engineer_features(batch: pd.DataFrame) -> pd.DataFrame:
        result = batch.copy()
        # Pressure deviation from mean (proxy for gradients)
        result["pressure_deviation"] = result["pressure"] - 1013.25
        # Temperature-humidity interaction
        result["temp_humidity"] = result["temperature"] * result["humidity"] / 100
        # Time features
        result["is_afternoon"] = ((result["hour"] >= 12) & (result["hour"] < 18)).astype(int)
        return result
    
    ds = ds.map_batches(engineer_features, batch_format="pandas")
    
    # Split
    train_ds, valid_ds = ds.train_test_split(test_size=0.2)
    
    # Train
    trainer = XGBoostTrainer(
        label_column="wind_speed",
        num_boost_round=100,
        params={
            "objective": "reg:squarederror",
            "eval_metric": ["rmse", "mae"],
            "max_depth": 6,
            "eta": 0.1,
            "subsample": 0.8,
        },
        datasets={"train": train_ds, "valid": valid_ds},
        scaling_config=ScalingConfig(num_workers=2),
    )
    
    return trainer.fit()

# Run solution
wind_result = build_wind_prediction_pipeline_solution()

print("\nWind Prediction Results:")
print(f"  RMSE: {wind_result.metrics.get('valid-rmse', 'N/A'):.4f}")
print(f"  MAE: {wind_result.metrics.get('valid-mae', 'N/A'):.4f}")

---

## 6. Comparison with Spark MLlib

| Aspect | Spark MLlib | Ray Train |
|--------|-------------|----------|
| Models | Limited built-in | XGBoost, PyTorch, TF, etc. |
| Data format | DataFrame | Ray Dataset |
| Distributed | Yes | Yes |
| GPU support | Limited | Native |
| Custom training | Limited | Full control |
| Best for | Simple models at scale | Complex ML pipelines |

---

## Summary

### Ray Train Key Concepts

1. **Trainers** - Pre-built for common frameworks
   - `XGBoostTrainer`, `LightGBMTrainer`
   - `TorchTrainer`, `TensorflowTrainer`

2. **ScalingConfig** - Control distributed training
   - `num_workers`: Number of training workers
   - `use_gpu`: Enable GPU training
   - `resources_per_worker`: CPU/GPU per worker

3. **Results** - Metrics and checkpoints
   - `result.metrics`: Training metrics
   - `result.checkpoint`: Saved model

### Pipeline Pattern

```
Data Source → Ray Data → Preprocessing → Train/Test Split → Ray Train → Metrics
```

### Next: Module 4

See `04_ray_spark_integration.md` for integrating Ray and Spark.

In [ ]:
# Cleanup
ray.shutdown()
print("Ray shutdown complete.")